# 快速入门 GPT-4 Vison 图像理解（GPT-4V）

从历史上看，语言模型系统仅接受**文本**作为输入。但是单一的输入形式，限制了大模型的应用落地范围。

随着技术发展，OpenAI 开发的 GPT-4 Turbo with Vision（简称 GPT-4V）允许模型接收**图像**作为输入，并回答关于它们的问题。

接收图片输入，实现图生文、OCR、场景理解，拓展了纯文本模型的边界.

## GPT-4V 核心知识点
    图像输入方式：支持 URL 图片、base64 编码图片两种输入格式
    能力边界：支持图片描述、OCR、图表解读、手写识别、场景推理
    限制说明：
    图片分辨率、大小有上限，高分辨率图片会自动下采样
   
## 智谱对应模型：GLM-4V（GLM-4 Vision）
    完全对标 GPT-4V 的图像理解能力，支持 URL 图片、本地图片输入
    支持中文场景优化，比如中文手写笔记识别、国内图表解读、场景描述
    调用方式和 OpenAI 高度兼容，可无缝迁移 GPT-4V 的学习代码

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
# 自动加载 .env 文件里的所有 key
load_dotenv()

client = OpenAI(
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL")
)


response = client.chat.completions.create(
    model="glm-4.1v-thinking-flashx",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "介绍下这幅图?"},
                {"type": "image_url", "image_url": {"url": "https://p6.itc.cn/q_70/images03/20200602/0c267a0d3d814c9783659eb956969ba1.jpeg"}}
            ]
        }
    ],
    max_tokens=300,
    temperature=0
)

print(response.choices[0].message.content)


<think>用户现在需要介绍这幅图，得先分析图中元素和对比点。首先看左右两边，左边是“16岁的我”，右边是“工作后的我”，都用柴犬形象但风格不同。

先看左边（16岁）：柴犬被P成肌肉发达、健壮的形象，文字有“16岁的我”“我前途一片光明”“身体素质高”“未来可期”“八九点钟的太阳”——这些表达年轻时的自信、活力，对未来充满希望，身体状态好，像朝气蓬勃的状态。

然后右边（工作后）：柴犬是普通坐着的形象，有点疲惫、没精神的样子，文字有“工作后的我”“好累好困 好想睡懒觉”“重大伤病不要靠近我啊啊啊”“我好弱小啊 啊啊啊啊啊”“我就是普通人”——这里表现出工作后的疲惫、缺乏精力，担心健康问题，自我感觉虚弱，和之前形成强烈对比。

还要解释这种对比的幽默和共鸣点：很多人从青春年少到工作后，状态变化大，用柴犬形象夸张表现这种落差，既搞笑又让人联想到自己经历的变化，属于网络流行的“人生对比” meme，用萌宠形象来直观展现成长或生活状态的转变，引发网友对“年轻时 vs 工作 后”的感慨，吐槽或自嘲生活压力带来的变化，同时用夸张的视觉差异（肌肉vs普通）和文字反差强化效果，让内容更生动有趣，容易传播


### 封装成一个函数 query_image_description

In [5]:
def query_image_description(url, prompt="介绍下这幅图?"):
    client = OpenAI(
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL")
)
    # 发送请求给 OpenAI 的聊天模型
    response = client.chat.completions.create(
       model="glm-4.1v-thinking-flashx",
        messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": url}}
            ]
        }
    ],
    max_tokens=300,
    temperature=0
    )
    
    # 返回模型的响应
    return response.choices[0].message.content

In [6]:
image_url = "https://p6.itc.cn/q_70/images03/20200602/0c267a0d3d814c9783659eb956969ba1.jpeg"
content = query_image_description(image_url)
print(content)

<think>用户现在需要介绍这幅图，得先分析图的元素和内涵。首先看画面结构：左右两边是“16岁的我”和“工作后的我”，用柴犬形象对比。

左边“16岁的我”：肌肉发达的柴犬，配文“我前途一片光明 未来可期 身体素质高 八九点钟的太阳”——这里用健壮形象代表年轻时的自信、活力，对未来充满希望，身体好像年轻人一样有朝气。

右边“工作后的我”：瘦弱的柴犬坐着，配文“好累好困 好想睡懒觉 重大伤病不要靠近我啊啊啊 我好弱小啊 啊啊啊啊啊啊 我就是普通人”——这里用消瘦疲惫的形象，表现工作后劳累、没精力，对健康担忧（怕伤病），自我感觉虚弱，从“未来可期”变成普通人的落差感。

然后整体是 meme 风格，用狗狗拟人化对比人生阶段的变化，幽默地表达从青春活力到职场疲惫、焦虑的情绪，反映很多人从学生时代到工作的状态变化，那种理想和现实的反差，以及对生活压力的感受。

现在组织语言，要说明图像内容（左右两只柴犬，文字标注）、对比点（形象、文字）、背后的情绪和主题（人生阶段变化，理想与现实，职场压力等）。

所以介绍的时候，先讲图片呈现的两部分（16岁和工作后，柴犬形象），再分别


#### 识别本地图像文件（Base64编码）
    GLM-4V 只能接收两种图片格式：
    图片 URL（公网可访问）
    Base64 编码字符串

In [11]:
from openai import OpenAI
import base64
import os

load_dotenv()

# 初始化智谱客户端
client = OpenAI(
     api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL")
)

# ----------------------
# 本地图片 → 转 Base64
# ----------------------
def image_to_base64(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

# ----------------------
# 调用 GLM-4V 识图
# ----------------------
def query_image_base64_description(image_path,text):
    base64_data = image_to_base64(image_path)
    
    response = client.chat.completions.create(
        model="glm-4.1v-thinking-flashx",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": text},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_data}"
                        }
                    }
                ]
            }
        ]
    )
    return response.choices[0].message.content



In [12]:
# 调用
content = query_image_base64_description("./images/105.jpg","请评价一下这篇作文")
print(content)

以下是针对这篇作文的评价，从立意、内容、表达等方面分析：  


### 优点部分  
1. **立意明确**：题目“拥抱保身 焕炼真诚精神”聚焦“真诚精神”这一主题，传递了积极的社会价值导向（强调在复杂环境下坚持真诚的品质），具有现实意义，体现了对“真诚”这一品质的重视。  
2. **书写态度**：整体书写较为工整，展现出作者认真完成作文的态度（尽管部分字迹因连笔或书写速度略显模糊，但不影响主要内容的理解）。  


### 需改进的部分  
1. **论据支撑不足**：文章在论述“拥抱保身、焕炼真诚精神”的过程中，较少结合具体事例、人物故事或生活场景来支撑观点，导致论证缺乏说服力和针对性，难以让读者深刻理解“追求精神”的具体实践路径。  
2. **语言与逻辑问题**：  
   - 句子表达有时不够精准，存在表意模糊的情况（可能与手写字迹辨识度、书写速度有关）；  
   - 段落过渡较为生硬，缺乏清晰的论证层次（如总分总、递进式等结构），导致文章条理感较弱，逻辑连贯性有所欠缺。  
3. **可读性待提升**：部分语句因书写原因或表达方式，可读性略受影响，一定程度上降低了内容的传播与理解效率。  


### 改进建议  
若作为学生练习作文，可尝试：  
- **补充具体论据**：结合名人名言、真实案例（如历史人物、身边事例）等，增强论证的说服力；  
- **优化结构**：通过合理划分段落、使用过渡句等方式，构建更清晰的论证层次（如“引出话题—分析意义—总结升华”的逻辑）；  
- **打磨语句**：精简冗余表达，确保语句通顺、表意准确，提升文章的可读性与逻辑性。  


总体来说，这篇作文有明确的主题方向，但需在论据、结构和表达细节上进一步优化，才能更好地实现“弘扬真诚精神”的核心意图。


In [13]:
# 调用
content = query_image_base64_description("./images/22893.jpg","请评价一下这篇作文")
print(content)

以下是针对这篇作文的评价，从结构、内容、表达等方面分析：  


### 优点分析  
1. **论点明确，结构清晰**  
   作文以“合理化求‘真’”为核心论题，开篇点明“求‘真’能引领社会发展，但非任何情境都适用”，随后通过“正反案例+辩证分析”展开，最后升华到“合理求真”的方向，整体结构有层次，逻辑连贯。  

2. **论证充分，案例恰当**  
   运用正反事例增强说服力：正面列举哥白尼（推动科学探索）、屠呦呦（医学领域求真）、袁隆平（农业领域求真）等事例，证明“合理求真”对社会发展的积极作用；反面列举苏格拉底（民主视野下的争议）、普罗米修斯（为人类造福却遭惩罚）等事例，说明“盲目求真”可能引发的问题。通过对比，凸显“合理化求真”的重要性，论证有力。  

3. **辩证思维，视角全面**  
   既肯定“求真”的历史价值（如推动认知、促进发展），也清醒认识其局限性（并非时时有效，可能引发冲突），并强调“合理化”的必要性（结合社会需求、坚守自我等），体现了辩证看待问题的思维，避免绝对化。  


### 可优化之处  
1. **语句通顺性与流畅度**  
   部分句子稍显生硬，如“在日常生活中，我们也常以求‘真’多变的态度面对一切，使得我们生活方方面面都处理完善。” 句子结构可调整得更自然；“对于国家重大机密，若想以为理学之思，去寻找、去探索时，是错误的。” 等表述可优化，让逻辑更顺畅。  

2. **案例与论点的贴合度**  
   部分事例虽经典，但可与“合理化”核心的关联可更紧密。例如普罗米修斯的例子，除“遭天罚”外，可更直接联系“求真”与时代局限性的关系，让论证更聚焦于“合理化”。  

3. **思想深度的拓展**  
   对“合理化”的具体标准（何时合理、不合理？判断依据是什么？）可进一步探讨，让文章思想更具延展性。目前虽提及“立足社会需求、坚守自我”等方向，但可补充更多理论支撑或现实分析，深化观点。  


### 总结  
这是一篇思路清晰、有辩证思维的议论文，展现了作者对“求真”问题的独立思考，具备较强的写作能力。经过语句优化、案例聚焦及思想深度的拓展后，文章的整体表现会更出色。


#### 在 Jupyter 标准输出中渲染 Markdown 格式内容

In [15]:
from IPython.display import display, Markdown

# 使用 display 和 Markdown 函数显示 Markdown 内容
display(Markdown(content))

以下是针对这篇作文的评价，从结构、内容、表达等方面分析：  


### 优点分析  
1. **论点明确，结构清晰**  
   作文以“合理化求‘真’”为核心论题，开篇点明“求‘真’能引领社会发展，但非任何情境都适用”，随后通过“正反案例+辩证分析”展开，最后升华到“合理求真”的方向，整体结构有层次，逻辑连贯。  

2. **论证充分，案例恰当**  
   运用正反事例增强说服力：正面列举哥白尼（推动科学探索）、屠呦呦（医学领域求真）、袁隆平（农业领域求真）等事例，证明“合理求真”对社会发展的积极作用；反面列举苏格拉底（民主视野下的争议）、普罗米修斯（为人类造福却遭惩罚）等事例，说明“盲目求真”可能引发的问题。通过对比，凸显“合理化求真”的重要性，论证有力。  

3. **辩证思维，视角全面**  
   既肯定“求真”的历史价值（如推动认知、促进发展），也清醒认识其局限性（并非时时有效，可能引发冲突），并强调“合理化”的必要性（结合社会需求、坚守自我等），体现了辩证看待问题的思维，避免绝对化。  


### 可优化之处  
1. **语句通顺性与流畅度**  
   部分句子稍显生硬，如“在日常生活中，我们也常以求‘真’多变的态度面对一切，使得我们生活方方面面都处理完善。” 句子结构可调整得更自然；“对于国家重大机密，若想以为理学之思，去寻找、去探索时，是错误的。” 等表述可优化，让逻辑更顺畅。  

2. **案例与论点的贴合度**  
   部分事例虽经典，但可与“合理化”核心的关联可更紧密。例如普罗米修斯的例子，除“遭天罚”外，可更直接联系“求真”与时代局限性的关系，让论证更聚焦于“合理化”。  

3. **思想深度的拓展**  
   对“合理化”的具体标准（何时合理、不合理？判断依据是什么？）可进一步探讨，让文章思想更具延展性。目前虽提及“立足社会需求、坚守自我”等方向，但可补充更多理论支撑或现实分析，深化观点。  


### 总结  
这是一篇思路清晰、有辩证思维的议论文，展现了作者对“求真”问题的独立思考，具备较强的写作能力。经过语句优化、案例聚焦及思想深度的拓展后，文章的整体表现会更出色。